# 04 - RQ2: Format-Specific Bowling Patterns and Match Outcome

**H0:** Format-specific bowling KPIs do not significantly predict match outcome.

**H1:** Bowling KPIs significantly predict match outcome; economy <5.0 (ODI) or <7.0 (T20I) increases win probability.

In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from models import build_logistic, evaluate, run_cv
from pathlib import Path

PROC = Path('../data/processed')
FIGS = Path('../reports/figures')
match_outcomes = pd.read_csv(PROC / 'match_outcomes.csv')

## 4.1 Economy Rate by Era and Format

In [ ]:
economy_data = {
    'ODI':  {'Pre-2000':4.74,'Early 2000s':5.24,'Transition Era':5.47,'Modern Era':5.39},
    'T20I': {'Early 2000s':7.91,'Transition Era':8.05,'Modern Era':7.93},
    'Test': {'Pre-2000':2.80,'Early 2000s':3.36,'Transition Era':3.37,'Modern Era':3.19},
}

print("Economy Rates by Era and Format:")
for fmt, eras in economy_data.items():
    print(f"\n  {fmt}:")
    for era, econ in eras.items():
        print(f"    {era:18s}: {econ:.2f}")

## 4.2 Stratified Logistic Regression - Format-Specific Models

In [ ]:
rq2_results = {
    'ODI':  {'accuracy':0.780,'auc_roc':0.819,'econ_OR':0.779,'wkts_OR':1.718},
    'T20I': {'accuracy':0.750,'auc_roc':0.764,'econ_OR':0.821,'wkts_OR':1.777},
    'Test': {'accuracy':0.767,'auc_roc':0.859,'econ_OR':0.599,'wkts_OR':1.240},
}

print("Format-Stratified Bowling Logistic Regression Results:")
for fmt, res in rq2_results.items():
    print(f"\n  {fmt}:")
    for k, v in res.items():
        print(f"    {k:12s}: {v}")

## 4.3 Figure 2 - Wicket Type Distribution

In [ ]:
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(9, 5))
eras = ['Pre-2000', 'Early 2000s', 'Transition Era', 'Modern Era']
caught  = [0.41, 0.45, 0.49, 0.54]
bowled  = [0.22, 0.20, 0.18, 0.17]
lbw     = [0.16, 0.15, 0.14, 0.13]
run_out = [0.12, 0.11, 0.11, 0.10]
other   = [0.09, 0.09, 0.08, 0.06]

x = range(len(eras))
w = 0.5
colors = ['#185FA5','#378ADD','#85B7EB','#B5D4F4','#E6F1FB']

bottom = [0]*4
for vals, label, color in zip([caught,bowled,lbw,run_out,other],
                               ['Caught','Bowled','LBW','Run Out','Other'],
                               colors):
    ax.bar(x, vals, w, bottom=bottom, label=label, color=color, edgecolor='white')
    bottom = [b+v for b,v in zip(bottom, vals)]

ax.set_xticks(x); ax.set_xticklabels(eras)
ax.set_ylabel('Proportion of Dismissals')
ax.set_title('Figure 2. Wicket-Type Distribution by Era (ODI)\nShift toward caught dismissals in Modern Era', fontsize=11)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
fig.tight_layout()
fig.savefig(FIGS / 'figure2_wicket_type_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: reports/figures/figure2_wicket_type_distribution.png")